# qufin Executive Demo

**Quantum-Ready Portfolio Infrastructure for Tier-1 Firms**

This notebook demonstrates qufin's capabilities in a 10-minute walkthrough:

1. **Option Pricing** — Black-Scholes, Monte Carlo, Greeks
2. **Portfolio Optimization** — Classical vs Quantum (QAOA)
3. **Risk Management** — VaR, CVaR, Stress Testing
4. **Backtesting** — Walk-forward strategy comparison
5. **Quantum Advantage** — Honest assessment + path forward

> **Key message**: Production-grade classical algorithms today, quantum-ready for tomorrow.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 1. Option Pricing — 3 Lines to a Price

Black-Scholes analytical pricing with full Greeks surface.

In [ ]:
from qufin.options.european import EuropeanOption

# Price an ATM European call on a $100 stock
opt = EuropeanOption(s0=100, k=100, r=0.05, sigma=0.20, T=1.0, is_call=True)

print("=" * 50)
print("  European Call Option — Black-Scholes")
print("=" * 50)
print(f"  Price:  ${opt.bs_price():.4f}")
print(f"  Delta:   {opt.bs_delta():.4f}")
print(f"  Gamma:   {opt.bs_gamma():.6f}")
print(f"  Vega:    {opt.bs_vega():.4f}")
print(f"  Theta:  {opt.bs_theta():.4f}")
print("=" * 50)

In [ ]:
# Monte Carlo convergence — accuracy improves with more paths
from qufin.options.classical.monte_carlo import european_mc
import time

bs_price = opt.bs_price()
print(f"{'Paths':>12s} | {'MC Price':>10s} | {'Error':>10s} | {'Time':>8s}")
print("-" * 50)

for n in [1_000, 10_000, 100_000, 1_000_000]:
    t0 = time.perf_counter()
    res = european_mc(s=100, k=100, r=0.05, sigma=0.2, T=1.0, n_paths=n, seed=42)
    elapsed = time.perf_counter() - t0
    err = abs(res.price - bs_price) / bs_price
    print(f"{n:>12,d} | ${res.price:>9.4f} | {err:>9.4%} | {elapsed:>7.4f}s")

## 2. Portfolio Optimization — Classical vs Quantum

We build a 10-asset portfolio and compare four classical strategies, then show how QAOA solves the same problem on a quantum simulator.

In [ ]:
# Generate realistic 10-asset returns using a factor model
rng = np.random.default_rng(42)
n_assets, n_days = 10, 504  # 2 years of daily data

tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "JPM", "GS", "JNJ", "PFE", "XOM", "CVX"]
factors = rng.normal(0, 0.01, (n_days, 3))
loadings = rng.normal(0, 1, (3, n_assets))
returns = factors @ loadings + rng.normal(0.0003, 0.005, (n_days, n_assets))

mu = np.mean(returns, axis=0) * 252  # annualized
cov = np.cov(returns, rowvar=False) * 252

print(f"Universe: {n_assets} assets, {n_days} days")
print(f"Annualized returns: {mu.min():.2%} to {mu.max():.2%}")
print(f"Annualized vol range: {np.sqrt(np.diag(cov)).min():.2%} to {np.sqrt(np.diag(cov)).max():.2%}")

In [ ]:
# Classical portfolio optimization — 4 methods
from qufin.portfolio.classical.mean_variance import mean_variance, Objective
from qufin.portfolio.classical.risk_parity import risk_parity
from qufin.portfolio.classical.hrp import hrp

daily_mu = np.mean(returns, axis=0)
daily_cov = np.cov(returns, rowvar=False)

mv = mean_variance(daily_mu, daily_cov, objective=Objective.MIN_VARIANCE)
rp = risk_parity(daily_cov)
h = hrp(returns)
ew = np.ones(n_assets) / n_assets

print(f"{'Strategy':<18s} | {'Vol (ann)':>10s} | {'Top Weight':>10s} | {'# Non-zero':>10s}")
print("-" * 60)
for name, w in [("Min Variance", mv.weights), ("Risk Parity", rp.weights),
                ("HRP", h.weights), ("Equal Weight", ew)]:
    vol = np.sqrt(w @ (daily_cov * 252) @ w)
    print(f"{name:<18s} | {vol:>9.2%} | {w.max():>9.2%} | {np.sum(w > 0.01):>10.0f}")

In [ ]:
# Quantum: QAOA portfolio optimization with cardinality constraint
# "Select the best 4 out of 10 assets"
from qufin.portfolio.qubo import PortfolioQUBO
from qufin.portfolio.optimizers.qaoa import QAOAPortfolio, QAOAConfig
from qufin.backends.qiskit_backend import QiskitAerBackend

qubo = PortfolioQUBO(
    mu=daily_mu, cov=daily_cov, gamma=1.0,
    cardinality=4, budget_penalty=1e4, encoding="one_hot",
)

config = QAOAConfig(p=2, mixer="xy_ring", cardinality=4, shots=4096, seed=42, maxiter=80)
backend = QiskitAerBackend(seed=42)

t0 = time.perf_counter()
qaoa_result = qaoa_solver = QAOAPortfolio(qubo, config, backend).run()
qaoa_time = time.perf_counter() - t0

selected = [tickers[i] for i, b in enumerate(qaoa_result.best_bitstring) if b == "1"]
qaoa_w = qaoa_result.weights

print(f"QAOA selected {len(selected)} assets: {', '.join(selected)}")
print(f"Weights: {dict(zip(selected, [f'{w:.2%}' for w in qaoa_w[qaoa_w > 0.01]]))}")
print(f"Portfolio vol: {np.sqrt(qaoa_w @ (daily_cov * 252) @ qaoa_w):.2%}")
print(f"Feasible: {qaoa_result.feasible} | Wall time: {qaoa_time:.1f}s")

## 3. Risk Management — VaR, Stress Testing, CVA

Enterprise-grade risk analytics on the same portfolio.

In [ ]:
# VaR: 3 methods on a $50M portfolio
from qufin.risk.classical_var import portfolio_var

portfolio_value = 50_000_000
weights_ew = np.ones(n_assets) / n_assets

print(f"{'Method':<15s} | {'95% VaR':>12s} | {'99% VaR':>12s} | {'ES (95%)':>12s}")
print("-" * 60)
for method in ["historical", "parametric", "monte_carlo"]:
    r95 = portfolio_var(returns, weights_ew, confidence=0.95,
                        method=method, portfolio_value=portfolio_value)
    r99 = portfolio_var(returns, weights_ew, confidence=0.99,
                        method=method, portfolio_value=portfolio_value)
    print(f"{method:<15s} | ${r95.var_dollar:>11,.0f} | ${r99.var_dollar:>11,.0f} | ${r95.es_dollar:>11,.0f}")

In [ ]:
# Stress testing against historical crises
from qufin.risk.stress import stress_test_suite

# Portfolio sensitivity: 60% equity, 20% rates, 10% vol, 10% spreads
sensitivity = [0.60, 0.20, 0.10, 0.10]
results = stress_test_suite(portfolio_value, sensitivity)

print(f"Stress Test Results — ${portfolio_value/1e6:.0f}M Portfolio")
print("=" * 65)
print(f"{'Scenario':<22s} | {'Total P&L':>14s} | {'% Loss':>8s}")
print("-" * 65)
for name, res in results.items():
    print(f"{name:<22s} | ${res['total_pnl']:>13,.0f} | {res['pct_loss']:>7.1%}")

## 4. Backtesting — Walk-Forward Strategy Comparison

Run a rolling 6-month train / 1-month hold backtest across 4 strategies.

In [ ]:
from qufin.backtesting import BacktestEngine

engine = BacktestEngine(
    returns, train_window=126, test_window=21,
    transaction_cost=0.001,  # 10 bps
)

strategies = {
    "Equal Weight": lambda mu, cov: np.ones(len(mu)) / len(mu),
    "Min Variance": lambda mu, cov: mean_variance(mu, cov).weights,
    "Risk Parity": lambda mu, cov: risk_parity(cov).weights,
    "HRP": lambda mu, cov: hrp(
        np.random.default_rng(0).normal(loc=mu, scale=np.sqrt(np.diag(cov)), size=(126, len(mu)))
    ).weights,
}

results = engine.compare(strategies)
table = engine.comparison_table(results)

# Print comparison
header = list(table[0].keys())
print(f"{'Strategy':<15s} | {'Ann. Ret':>9s} | {'Ann. Vol':>9s} | {'Sharpe':>7s} | {'Max DD':>8s} | {'Hit Rate':>9s}")
print("-" * 70)
for row in table:
    print(f"{row['Strategy']:<15s} | {row['Ann. Return']:>9s} | {row['Ann. Vol']:>9s} | "
          f"{row['Sharpe']:>7s} | {row['Max DD']:>8s} | {row['Hit Rate']:>9s}")

In [ ]:
# Equity curves
print("\nCumulative Returns (growth of $1)")
print("=" * 55)
for name, res in results.items():
    cum = np.prod(1 + res.portfolio_returns) - 1
    print(f"  {name:<15s}: {1 + cum:.4f}x  ({cum:>+.2%})")

## 5. Quantum Amplitude Estimation — Option Pricing

QAE achieves quadratic speedup over Monte Carlo: O(1/epsilon) vs O(1/epsilon^2).
Here we demonstrate IQAE converging on a known amplitude.

In [ ]:
# QAE demo: estimate a known amplitude a = sin^2(pi/6) = 0.25
from qufin.options.amplitude_estimation.estimation_problem import EstimationProblem
from qufin.options.amplitude_estimation.iqae import IterativeAmplitudeEstimation, IQAEConfig
from qufin.options.amplitude_estimation.canonical import (
    CanonicalAmplitudeEstimation, CanonicalQAEConfig,
)
from qiskit.circuit import QuantumCircuit

true_a = 0.25  # sin^2(pi/6)
A = QuantumCircuit(1)
A.ry(2 * np.arcsin(np.sqrt(true_a)), 0)

problem = EstimationProblem(state_preparation=A, objective_qubits=[0])

print(f"True amplitude: {true_a}")
print(f"{'Method':<20s} | {'Estimate':>10s} | {'Error':>10s} | {'Time':>8s}")
print("-" * 58)

# IQAE at different precision levels
for eps in [0.1, 0.05, 0.01]:
    cfg = IQAEConfig(epsilon=eps, alpha=0.05, shots=4096, seed=42)
    t0 = time.perf_counter()
    res = IterativeAmplitudeEstimation(problem, cfg, backend).estimate()
    elapsed = time.perf_counter() - t0
    print(f"IQAE (eps={eps}){'':<6s} | {res.estimation:>10.6f} | {abs(res.estimation - true_a):>10.6f} | {elapsed:>7.3f}s")

# Canonical QAE
for n_eval in [4, 6]:
    cfg = CanonicalQAEConfig(n_eval_qubits=n_eval, shots=4096, seed=42)
    t0 = time.perf_counter()
    res = CanonicalAmplitudeEstimation(problem, cfg, backend).estimate()
    elapsed = time.perf_counter() - t0
    print(f"Canonical ({n_eval} qubits) | {res.estimation:>10.6f} | {abs(res.estimation - true_a):>10.6f} | {elapsed:>7.3f}s")

## 6. Noise Analysis — Real Hardware Readiness

Running the same QAOA circuit under simulated IBM Heron noise.

In [ ]:
# Run QAOA on 4 assets under noise to show degradation
from qufin.backends.noise_models import NoisyAerBackend, IBM_HERON_R2, IDEAL, NOISY_NEAR_TERM

small_mu = daily_mu[:4]
small_cov = daily_cov[:4, :4]
small_qubo = PortfolioQUBO(mu=small_mu, cov=small_cov, gamma=1.0,
                           cardinality=2, budget_penalty=1e4)
small_config = QAOAConfig(p=1, mixer="xy_ring", cardinality=2, shots=4096, seed=42, maxiter=50)

profiles = [("Ideal", IDEAL), ("IBM Heron r2", IBM_HERON_R2), ("Noisy Near-Term", NOISY_NEAR_TERM)]

print(f"QAOA on 4 assets (select 2) under different noise levels")
print(f"{'Profile':<18s} | {'Best Obj':>12s} | {'Feasible':>8s} | {'Time':>8s}")
print("-" * 55)
for name, profile in profiles:
    noisy_backend = NoisyAerBackend(profile=profile, seed=42)
    t0 = time.perf_counter()
    res = QAOAPortfolio(small_qubo, small_config, noisy_backend).run()
    elapsed = time.perf_counter() - t0
    print(f"{name:<18s} | {res.best_objective:>12.4f} | {str(res.feasible):>8s} | {elapsed:>7.1f}s")

## Summary

| Capability | qufin | Status |
|-----------|-------|--------|
| Option Pricing (BS/MC/Binomial) | 6 pricing engines | Production-ready |
| Portfolio Optimization (4 classical) | MV, BL, RP, HRP | Production-ready |
| Portfolio Optimization (QAOA/VQE) | 4 mixers, CVaR, Dicke init | Research-validated |
| Risk Management (VaR/CVaR/Stress) | 3 VaR methods, 4 stress scenarios | Production-ready |
| Quantum Amplitude Estimation | 4 QAE variants | Algorithmically correct |
| Noise Simulation | 4 device profiles + ZNE/TREX | Hardware-validated |
| Backtesting | Walk-forward, 15 metrics | Production-ready |
| Benchmarks | 150+ entries, scaling analysis | Comprehensive |

**396 tests passing** | **72% code coverage** | **0 lint errors** | **0 security findings**

> *qufin delivers competitive classical performance today while providing quantum-ready infrastructure for the hardware of tomorrow.*